# 15-Minute Temperature Resampling

This notebook resamples 10-minute temperature data to 15-minute resolution.

In [1]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path

### Load and Combine Temperature Files

In [2]:
# find all Helsinki-Vantaa airport temperature CSV files (10-min raw data)
# NOTE: cwd = notebook directory (Temperature folder)
temp_dir = Path('.')
temp_files = sorted(temp_dir.glob('Helsinki-Vantaa airport_*.csv'))

print(f'Found {len(temp_files)} temperature files:')
for f in temp_files:
    print(f'  {f.name}')


Found 7 temperature files:
  Helsinki-Vantaa airport_ 1.1.2023 - 30.6.2023.csv
  Helsinki-Vantaa airport_ 1.1.2024 - 30.6.2024.csv
  Helsinki-Vantaa airport_ 1.1.2025 - 30.6.2025.csv
  Helsinki-Vantaa airport_ 1.11.2024 - 31.12.2024.csv
  Helsinki-Vantaa airport_ 1.7.2023 - 31.12.2023.csv
  Helsinki-Vantaa airport_ 1.7.2024 - 31.10.2024.csv
  Helsinki-Vantaa airport_ 1.7.2025 - 31.12.2025.csv


In [3]:
# read and combine all temperature files into one DataFrame
temp_list = []
for f in temp_files:
    tmp = pd.read_csv(f)
    # build a full datetime from Year/Month/Day/Time columns
    tmp['datetime'] = pd.to_datetime(
        tmp['Year'].astype(str) + '-' +
        tmp['Month'].astype(str).str.zfill(2) + '-' +
        tmp['Day'].astype(str).str.zfill(2) + ' ' +
        tmp['Time [Local time]'].astype(str)
    )
    # mark as Helsinki local time
    tmp['datetime'] = tmp['datetime'].dt.tz_localize('Europe/Helsinki', ambiguous='NaT', nonexistent='NaT')
    # keep only the columns we need, and force temperature to numeric
    tmp = tmp[['datetime', 'Air temperature mean [°C]']].rename(
        columns={'Air temperature mean [°C]': 'temperature_c'}
    )
    tmp['temperature_c'] = pd.to_numeric(tmp['temperature_c'], errors='coerce')
    temp_list.append(tmp)

temp_df = pd.concat(temp_list, ignore_index=True)
temp_df = temp_df.sort_values('datetime').drop_duplicates(subset=['datetime']).reset_index(drop=True)

print('Temperature combined shape:', temp_df.shape)
print('Date range:', temp_df['datetime'].min(), '->', temp_df['datetime'].max())
temp_df.head(3)

Temperature combined shape: (157789, 2)
Date range: 2023-01-01 00:00:00+02:00 -> 2025-12-31 23:50:00+02:00


,datetime,temperature_c
0,2023-01-01 00:00:00+02:00,4.4
1,2023-01-01 00:10:00+02:00,4.5
2,2023-01-01 00:20:00+02:00,4.5


### Resample Temperature to 15-Minute

In [4]:
# set datetime as index for resampling
temp_10min = temp_df.set_index('datetime')

# resample temperature from 10-min to 15-min using mean
# 15min is NOT a multiple of 10min, so some buckets get 1 row, others get 2
temp_15min = temp_10min.resample('15min').mean()

print('Temperature 15-min shape:', temp_15min.shape)
print('NaN count:', temp_15min['temperature_c'].isna().sum())
temp_15min.head(6)

Temperature 15-min shape: (105216, 1)
NaN count: 48


,temperature_c
datetime,
2023-01-01 00:00:00+02:00,4.45
2023-01-01 00:15:00+02:00,4.50
2023-01-01 00:30:00+02:00,4.45
2023-01-01 00:45:00+02:00,4.30
2023-01-01 01:00:00+02:00,4.30
2023-01-01 01:15:00+02:00,4.20


### Save Resampled 15-Minute Weather Files

In [5]:
# save temperature 15-min (cwd = notebook folder, so relative path works)
temp_15min.to_csv('temperature_15min.csv', index=True)
print('Saved: temperature_15min.csv')

Saved: temperature_15min.csv


In [6]:
print('\n=== Summary ===')

print(f'Temperature: {len(temp_15min)} rows, {temp_15min.index.min()} to {temp_15min.index.max()}')


=== Summary ===
Temperature: 105216 rows, 2023-01-01 00:00:00+02:00 to 2025-12-31 23:45:00+02:00
